In [0]:
import os
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, StringType

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_DATABRICKS:
    catalog = "visagio_atividade_v_energy_360"
    bronze_schema = f"{catalog}.bronze"
    silver_schema = f"{catalog}.silver"
else:
    from local_runtime import setup_local_env
    spark, dbutils, display, _paths, _schemas = setup_local_env()
    catalog = _schemas["catalog"]
    bronze_schema = _schemas["bronze"]
    silver_schema = _schemas["silver"]

print(f"Iniciando a transformação para a camada Silver no schema: {silver_schema}")

df_uc_bronze = spark.table(f"{bronze_schema}.bronze_unidades_consumidoras")
df_leituras_bronze = spark.table(f"{bronze_schema}.bronze_leituras")
df_medidores_bronze = spark.table(f"{bronze_schema}.bronze_medidores")
df_gd_bronze = spark.table(f"{bronze_schema}.bronze_geracao_distribuida")

Iniciando a transformação para a camada Silver no schema: visagio_atividade_v_energy_360.silver


In [0]:
# Fase 2: Higienização e Enriquecimento

def clean_double(col_name):
    return F.expr(f"try_cast(regexp_replace(trim(cast({col_name} as string)), ',', '.') as double)")

def safe_date_parse_v2(col_name):
    return F.expr(f"""
        coalesce(
            try_to_date(cast({col_name} as string), 'yyyy-MM-dd'),
            try_to_date(cast({col_name} as string), 'yyyy/MM/dd'), 
            try_to_date(cast({col_name} as string), 'dd/MM/yyyy'),
            try_to_date(cast({col_name} as string), 'dd.MM.yyyy'),
            try_to_date(cast({col_name} as string), 'MM-dd-yyyy'),
            try_to_date(cast({col_name} as string), 'dd-MM-yyyy')
        )
    """)

@F.udf(returnType=StringType())
def fix_mojibake(text):
    if text is None:
        return None
    try:
        return text.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text

def clean_string(col_name):
    accents = 'ÀÁÂÃÄÅÇÈÉÊËÌÍÎÏÑÒÓÔÕÖÙÚÛÜÝàáâãäåçèéêëìíîïñòóôõöùúûüý'
    no_accents = 'AAAAAACEEEEIIIINOOOOOUUUUYaaaaaaceeeeiiiinooooouuuuy'
    repaired_col = fix_mojibake(F.col(col_name))
    return F.upper(F.trim(F.translate(repaired_col, accents, no_accents)))

# 1. Padronização e Tipagem
df_uc_silver = df_uc_bronze \
    .filter(
        F.col("id_uc").isNotNull() & 
        (F.lower(F.trim(F.col("id_uc"))) != "id_uc") & 
        F.trim(F.col("id_uc")).rlike(r"^UC[0-9]{6}$")
    ) \
    .withColumn("id_uc", F.trim(F.col("id_uc"))) \
    .withColumn("cpf_cnpj", F.regexp_replace(F.col("cpf_cnpj").cast("string"), r"\D", "")) \
    .withColumn("nome_titular", clean_string("nome_titular")) \
    .withColumn("logradouro", clean_string("logradouro")) \
    .withColumn("numero", 
                F.when(F.col("numero").cast("int") < 0, F.lit(None))
                 .otherwise(F.col("numero").cast("int"))) \
    .withColumn("bairro", clean_string("bairro")) \
    .withColumn("municipio", clean_string("municipio")) \
    .withColumn("uf", F.when(F.upper(F.trim(F.regexp_replace(F.col("uf"), r"\.", ""))) == "PERNAMBUCO", "PE")
                   .otherwise(F.upper(F.trim(F.regexp_replace(F.col("uf"), r"\.", ""))))) \
    .withColumn("cep_limpo", F.regexp_replace(F.col("cep").cast("string"), r"\D", "")) \
    .withColumn("cep", 
                F.when(F.length(F.col("cep_limpo")) == 8, F.col("cep_limpo"))
                 .otherwise(F.lit(None))) \
    .drop("cep_limpo") \
    .withColumn("classe", clean_string("classe")) \
    .withColumn("subgrupo_tarifario", F.upper(F.trim(F.col("subgrupo_tarifario")))) \
    .withColumn("alimentador", F.upper(F.trim(F.col("alimentador")))) \
    .withColumn("data_ligacao", safe_date_parse_v2("data_ligacao")) \
    .withColumn("status", F.upper(F.trim(F.col("status")))) \
    .withColumn("latitude", F.col("latitude").cast("double")) \
    .withColumn("longitude", F.col("longitude").cast("double"))

df_leituras_silver = df_leituras_bronze \
    .filter(F.col("id_leitura").isNotNull() & F.col("id_uc").isNotNull()) \
    .withColumn("id_leitura", F.trim(F.col("id_leitura"))) \
    .withColumn("id_uc", F.trim(F.col("id_uc"))) \
    .withColumn("id_medidor", 
                F.when(F.col("id_medidor").isNull(), F.lit("DESCONHECIDO"))
                 .otherwise(F.trim(F.col("id_medidor")))) \
    .withColumn("mes_referencia", F.expr("try_to_date(trim(mes_referencia), 'yyyy-MM')")) \
    .withColumn("data_leitura_str", F.trim(F.col("data_leitura").cast("string"))) \
    .withColumn("data_leitura", F.expr("""
        CASE
          WHEN try_to_date(data_leitura_str, 'yyyy-MM-dd') IS NOT NULL THEN try_to_date(data_leitura_str, 'yyyy-MM-dd')
          WHEN try_to_date(data_leitura_str, 'dd/MM/yyyy') IS NOT NULL THEN try_to_date(data_leitura_str, 'dd/MM/yyyy')
          WHEN try_to_date(data_leitura_str, 'dd.MM.yyyy') IS NOT NULL THEN try_to_date(data_leitura_str, 'dd.MM.yyyy')
          WHEN date_format(try_to_date(data_leitura_str, 'dd-MM-yyyy'), 'yyyy-MM') = date_format(mes_referencia, 'yyyy-MM') THEN try_to_date(data_leitura_str, 'dd-MM-yyyy')
          WHEN date_format(try_to_date(data_leitura_str, 'MM-dd-yyyy'), 'yyyy-MM') = date_format(mes_referencia, 'yyyy-MM') THEN try_to_date(data_leitura_str, 'MM-dd-yyyy')
          ELSE coalesce(try_to_date(data_leitura_str, 'dd-MM-yyyy'), try_to_date(data_leitura_str, 'MM-dd-yyyy'))
        END
    """)) \
    .drop("data_leitura_str") \
    .withColumn("leitura_anterior", 
                F.when(clean_double("leitura_anterior") < 0, F.lit(None))
                 .otherwise(clean_double("leitura_anterior"))) \
    .withColumn("leitura_atual", 
                F.when(clean_double("leitura_atual") < 0, F.lit(None))
                 .otherwise(clean_double("leitura_atual"))) \
    .withColumnRenamed("consumo_kwh", "consumo_apurado_kwh") \
    .withColumn("consumo_apurado_kwh", 
                F.when(clean_double("consumo_apurado_kwh") < 0, F.lit(None))
                 .otherwise(clean_double("consumo_apurado_kwh"))) \
    .withColumn("tipo_leitura", F.upper(F.trim(F.col("tipo_leitura"))))

df_medidores_silver = df_medidores_bronze \
    .filter(F.col("id_medidor").isNotNull() & F.col("id_uc").isNotNull()) \
    .withColumn("id_medidor", F.trim(F.col("id_medidor"))) \
    .withColumn("id_uc", F.trim(F.col("id_uc"))) \
    .withColumn("modelo", clean_string("modelo")) \
    .withColumn("tipo", clean_string("tipo")) \
    .withColumn("tipo", F.regexp_replace(F.col("tipo"), "-", "")) \
    .withColumn("tipo", F.when(F.col("tipo") == "SMART METER", "SMART").otherwise(F.col("tipo"))) \
    .withColumnRenamed("data_instalacao", "data_instalacao_medidor") \
    .withColumn("data_instalacao_medidor", safe_date_parse_v2("data_instalacao_medidor")) \
    .withColumn("data_retirada_temp", safe_date_parse_v2("data_retirada")) \
    .withColumn("data_retirada", 
                F.when(F.col("data_retirada_temp").cast("string") == "9999-12-31", F.lit(None))
                 .otherwise(F.col("data_retirada_temp"))) \
    .drop("data_retirada_temp") \
    .withColumn("data_ultima_manutencao", safe_date_parse_v2("data_ultima_manutencao")) \
    .withColumn("status", clean_string("status"))

df_gd_silver = df_gd_bronze \
    .filter(F.col("id_uc").isNotNull()) \
    .withColumn("id_uc", F.trim(F.col("id_uc"))) \
    .withColumnRenamed("potencia_kwp", "potencia_instalada_kw") \
    .withColumn("potencia_instalada_kw", 
                F.when(clean_double("potencia_instalada_kw") < 0, F.lit(None))
                 .otherwise(clean_double("potencia_instalada_kw"))) \
    .withColumnRenamed("data_conexao", "data_conexao_gd") \
    .withColumn("data_conexao_gd", safe_date_parse_v2("data_conexao_gd")) \
    .withColumn("modalidade", clean_string("modalidade")) \
    .withColumn("energia_injetada_kwh_media_mensal", 
                F.when(clean_double("energia_injetada_kwh_media_mensal") < 0, F.lit(None))
                 .otherwise(clean_double("energia_injetada_kwh_media_mensal")))

In [0]:
# 2. Deduplicação Canônica

from pyspark.sql.window import Window

campos_cadastrais = ["nome_titular", "logradouro", "numero", "bairro", "municipio", "uf", "cep",
                     "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "status",
                     "latitude", "longitude"]
completude_cadastral = sum(F.col(campo).isNotNull().cast("int") for campo in campos_cadastrais)
janela_uc = Window.partitionBy("id_uc").orderBy(
    completude_cadastral.desc(), F.col("data_ligacao").desc_nulls_last(),
    F.col("_ingested_at").desc(), F.col("_source_file").desc())
df_uc_silver = df_uc_silver.withColumn("rn", F.row_number().over(janela_uc)).filter("rn = 1").drop("rn")

# Deduplicação determinística de leituras usando Window
medidores_periodo = df_medidores_silver.select(
    "id_uc", "id_medidor",
    F.col("data_instalacao_medidor").alias("inicio_medidor"),
    F.col("data_retirada").alias("fim_medidor"))

df_leituras_silver = df_leituras_silver.alias("lei") \
    .join(medidores_periodo.alias("med"),
          (F.col("lei.id_uc") == F.col("med.id_uc")) & (F.col("lei.id_medidor") == F.col("med.id_medidor")),
          "left") \
    .withColumn("medidor_vigente", F.coalesce(
        (F.col("lei.data_leitura") >= F.col("med.inicio_medidor"))
        & (F.col("med.fim_medidor").isNull() | (F.col("lei.data_leitura") <= F.col("med.fim_medidor"))),
        F.lit(False)).cast("int")) \
    .select("lei.*", "medidor_vigente")

janela = Window.partitionBy("id_uc", "mes_referencia") \
               .orderBy(F.col("medidor_vigente").desc(),
                        F.col("data_leitura").desc_nulls_last(), F.col("id_leitura").desc())

df_leituras_silver = df_leituras_silver \
    .withColumn("rn", F.row_number().over(janela)) \
    .filter("rn = 1") \
    .drop("rn", "medidor_vigente")

df_medidores_silver = df_medidores_silver.dropDuplicates(["id_medidor", "id_uc", "data_instalacao_medidor"])

janela_gd = Window.partitionBy("id_uc").orderBy(
    F.col("data_conexao_gd").desc_nulls_last(), F.col("potencia_instalada_kw").desc_nulls_last())
df_gd_silver = df_gd_silver.withColumn("rn", F.row_number().over(janela_gd)).filter("rn = 1").drop("rn")

In [0]:
# 3. Enriquecimento de GD Solar (no Cadastro)
df_uc_enriquecida = df_uc_silver.alias("uc") \
    .join(df_gd_silver.alias("gd"), "id_uc", "left") \
    .withColumn("possui_geracao_solar", F.col("gd.potencia_instalada_kw").isNotNull()) \
    .withColumn("potencia_solar_kw", F.when(F.col("possui_geracao_solar"), F.col("gd.potencia_instalada_kw")).otherwise(0.0)) \
    .select("uc.*", "possui_geracao_solar", "potencia_solar_kw")

In [0]:
# 4. Integridade Referencial & Órfãos
df_leituras_orfaos = df_leituras_silver.join(df_uc_enriquecida, "id_uc", "left_anti")
count_orfaos = df_leituras_orfaos.count()
print(f"AUDITORIA: Foram encontradas {count_orfaos} leituras órfãs (id_uc inexistente no cadastro).")

# Isola os órfãos e registra a métrica de auditoria
df_leituras_orfaos.withColumn("_auditado_em", F.current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_leituras_orfaos")

spark.createDataFrame([("silver_leituras", "id_uc inexistente no cadastro", count_orfaos)],
                      ["tabela", "regra", "registros"]) \
    .withColumn("auditado_em", F.current_timestamp()) \
    .write.format("delta").mode("append").saveAsTable(f"{silver_schema}.auditoria_qualidade")

df_leituras_validas = df_leituras_silver.join(df_uc_enriquecida.select("id_uc"), "id_uc", "inner")

AUDITORIA: Foram encontradas 0 leituras órfãs (id_uc inexistente no cadastro).


In [0]:
# 5. Tratamento de Anomalias de Medição (Zero Abstração)
df_trocas_medidor = df_medidores_silver \
    .filter(F.col("data_retirada").isNotNull()) \
    .withColumn("mes_referencia_troca", F.trunc(F.col("data_retirada"), "month")) \
    .select("id_uc", "mes_referencia_troca") \
    .dropDuplicates(["id_uc", "mes_referencia_troca"])

df_leituras_tratadas = df_leituras_validas \
    .join(df_trocas_medidor, 
          (df_leituras_validas.id_uc == df_trocas_medidor.id_uc) & 
          (df_leituras_validas.mes_referencia == df_trocas_medidor.mes_referencia_troca), 
          "left") \
    .drop(df_trocas_medidor.id_uc) \
    .withColumn("houve_troca_medidor", F.col("mes_referencia_troca").isNotNull()) \
    .withColumn("is_rollover", 
                F.coalesce(
                    (F.col("leitura_atual") < F.col("leitura_anterior")) & 
                    (F.col("leitura_anterior") >= 90000) & 
                    (F.col("leitura_atual") <= 10000) & 
                    (~F.col("houve_troca_medidor")),
                    F.lit(False)
                )) \
    .withColumn("anomalia_leitura", 
                F.coalesce(
                    (F.col("leitura_atual") < F.col("leitura_anterior")) & 
                    (~F.col("is_rollover")) & 
                    (~F.col("houve_troca_medidor")), 
                    F.lit(False)
                )) \
    .withColumn("limite_contador", F.pow(F.lit(10), F.length(F.col("leitura_anterior").cast("int").cast("string")))) \
    .withColumn("consumo_apurado_kwh", 
                F.when(
                    F.col("is_rollover"), 
                    (F.col("limite_contador") - F.col("leitura_anterior")) + F.col("leitura_atual")
                ).otherwise(F.col("consumo_apurado_kwh"))) \
    .withColumn("consumo_apurado_kwh", 
                F.when(F.col("consumo_apurado_kwh") < 0, F.lit(None))
                 .otherwise(F.col("consumo_apurado_kwh"))) \
    .withColumn("consumo_imputado_leitura", F.coalesce(
        F.col("consumo_apurado_kwh").isNull() & (~F.col("anomalia_leitura"))
        & (F.col("leitura_atual") >= F.col("leitura_anterior")), F.lit(False))) \
    .withColumn("consumo_apurado_kwh", F.when(F.col("consumo_imputado_leitura"),
        F.col("leitura_atual") - F.col("leitura_anterior")).otherwise(F.col("consumo_apurado_kwh"))) \
    .drop("mes_referencia_troca", "limite_contador")

In [0]:
# 6. Gravação Final na Camada Silver
print("Salvando as tabelas tratadas na Camada Silver...")

df_uc_enriquecida.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_unidades_consumidoras")

df_leituras_tratadas.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_leituras")

df_medidores_silver.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_medidores")

df_gd_silver.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_geracao_distribuida")

total_anomalias = spark.table(f"{silver_schema}.silver_leituras").filter("anomalia_leitura").count()
spark.createDataFrame(
    [("silver_leituras", "anomalia_leitura (leitura caiu sem rollover nem troca)", total_anomalias)],
    ["tabela", "regra", "registros"]) \
    .withColumn("auditado_em", F.current_timestamp()) \
    .write.format("delta").mode("Overwrite").saveAsTable(f"{silver_schema}.auditoria_qualidade")

print("Fase 2: Ingestão na Camada Silver concluída com sucesso.")

Salvando as tabelas tratadas na Camada Silver...
Fase 2: Ingestão na Camada Silver concluída com sucesso.
